# CarePath-AI | Phase 02 - Data Engineering & Patient Survey
**Release v0.2.0 · Synthetic research only · Phase 1 v0.1.1 retained**

Build 50,000 longitudinal fictional appointments, validate data quality and chronology, persist to the integrated SQLite schema, produce two train-only preprocessing pipelines, examine synthetic treatment effects, and demonstrate the local survey. No model training or real healthcare actions. Execute from the repository root or notebooks directory; see README. The historical offer policy is not a real treatment eligibility policy. Oracle data are evaluation-only.

**Licence:** Apache-2.0 · Copyright 2026 Samudra De Silva (project-owned material). See `LICENSE` and `NOTICE` in the repository. AI-assisted development is acknowledged in `NOTICE`.

**Research disclaimer:** CarePath-AI is an experimental research and software engineering prototype using synthetic healthcare data. It has not been clinically validated and is not intended for clinical diagnosis, medical advice, autonomous healthcare decisions or actual patient management. It is not affiliated with or endorsed by the NHS. Real-world deployment requires appropriate regulatory, ethical, clinical, privacy and security assessment and authorisation. This statement is separate from the Apache License 2.0 warranty and liability provisions.

## 1. Environment and configuration

In [1]:
from pathlib import Path
import sys, os, json, platform, sqlite3, subprocess, re, tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib, sklearn, matplotlib
from IPython.display import display, Markdown
cwd=Path.cwd().resolve()
ROOT=next((x for x in [cwd,*cwd.parents] if (x/'carepath/phase2.py').is_file()),None)
if ROOT is None: raise FileNotFoundError('Open this notebook inside the CarePath-AI repository or extracted release.')
sys.path.insert(0,str(ROOT))
from carepath.phase2 import (generate,validate,splits,preprocessor,integrate,verify_database,submit_survey,survey_widget,dataset_hash,SEED,N,ARMS,BASE,FEATURES,BARRIERS,TRAIN_END,VALID_END)
from carepath.foundation import initialize, connect, seed_demo, integrity
DATA=ROOT/'data'; REPORTS=ROOT/'reports/phase2'; MODELS=ROOT/'models'; DB=ROOT/'database/carepath_ai_phase2.db'
for folder in [DATA,DATA/'splits',REPORTS,MODELS]: folder.mkdir(parents=True,exist_ok=True)
# Never leave an earlier PASS report if this run fails.
(REPORTS/'validation.json').write_text(json.dumps({'status':'RUNNING_OR_INCOMPLETE'},indent=2),encoding='utf-8')
print('Python',platform.python_version(),'SQLite',sqlite3.sqlite_version,'NumPy',np.__version__,'Pandas',pd.__version__,'scikit-learn',sklearn.__version__)
print('Project root:',ROOT)

Python 3.13.5 SQLite 3.46.1 NumPy 2.3.5 Pandas 2.2.3 scikit-learn 1.8.0
Project root: /mnt/data/CarePath_AI_Phase2_v0.2.0


## 2. Generate reproducible longitudinal appointments
All six barrier variables are modeled with correlations to fictional patient/appointment characteristics. Outcome and intervention assignment use a known simulation mechanism. Repeated patients have a minimum seven-day visit spacing. The index is 48 hours before the appointment; only active appointments enter binary prediction. See methodology for the full design.

In [2]:
df,oracle=generate(SEED)
checks=validate(df,oracle)
display(pd.DataFrame({'check':list(checks),'passed':list(checks.values())}))
assert all(checks.values()),checks
print(f'{len(df):,} appointments; {df.patient_id.nunique():,} unique fictional patients')
print('Fingerprint:',dataset_hash(df))
display(df.head(5))

,check,passed
0,exactly_50000,True
1,unique_appointments,True
2,12500_patients,True
3,valid_patient_appointment_counts,True
4,no_real_identifiers,True
5,valid_dispositions,True
6,binary_target_only_active,True
7,competing_dispositions_excluded,True
8,survey_missingness_preserved,True
9,survey_timestamps_pre_index,True


50,000 appointments; 12,500 unique fictional patients


Fingerprint: 0ee4f019f02df0dc257a4f364197fca7448ed9c5c04faff5bf08d42f73739a2e


,appointment_id,patient_id,scheduled_at,booked_at,index_at,survey_at,age,age_band,distance_km,lead_days,prior_appointments,prior_no_shows,prior_cancellations,prior_no_show_rate,appointment_hour,weekday,month,sms_opt_in,vehicle_available,area_deprivation_proxy,department,appointment_type,employment,preferred_channel,survey_responded,attendance_confidence,support_consent,active_at_index,treatment,outcome,no_show,transport_difficulty,financial_difficulty,accessibility_need,timing_conflict,language_support,caring_responsibility,split
0,R-A-00036-1,R-P-00036,2023-01-02T08:00:00Z,2022-12-08T08:00:00Z,2022-12-31T08:00:00Z,2022-12-30T08:00:00Z,27,18-29,5.6,25,0,0,0,0.0,8,0,1,1,1,3,Cardiology,Follow-up,not_employed,sms,1,8.0,1.0,1,reminder_offer,no_show,1.0,1.0,0.0,0.0,0.0,0.0,0.0,train
1,R-A-03074-1,R-P-03074,2023-01-02T08:00:00Z,2022-10-18T08:00:00Z,2022-12-31T08:00:00Z,2022-12-30T08:00:00Z,46,45-59,4.6,76,0,0,0,0.0,8,0,1,1,1,3,Cardiology,New,not_employed,sms,1,9.0,1.0,1,scheduling_offer,no_show,1.0,0.0,0.0,0.0,0.0,0.0,0.0,train
2,R-A-06854-1,R-P-06854,2023-01-02T08:00:00Z,2022-12-17T08:00:00Z,2022-12-31T08:00:00Z,2022-12-30T08:00:00Z,34,30-44,9.4,16,0,0,0,0.0,8,0,1,1,1,3,Physiotherapy,New,not_employed,sms,1,5.0,1.0,1,transport_offer,no_show,1.0,1.0,0.0,0.0,0.0,0.0,0.0,train
3,R-A-07238-1,R-P-07238,2023-01-02T08:00:00Z,2022-10-10T08:00:00Z,2022-12-31T08:00:00Z,2022-12-30T08:00:00Z,18,18-29,26.9,84,0,0,0,0.0,8,0,1,1,0,5,Physiotherapy,Follow-up,employed,sms,1,7.0,0.0,1,reminder_offer,attended,0.0,0.0,0.0,1.0,0.0,0.0,1.0,train
4,R-A-03388-1,R-P-03388,2023-01-02T08:30:00Z,2022-12-09T08:30:00Z,2022-12-31T08:30:00Z,2022-12-30T08:30:00Z,25,18-29,8.2,24,0,0,0,0.0,8,0,1,1,1,5,Cardiology,Follow-up,employed,sms,1,8.0,1.0,1,usual_care,attended,0.0,0.0,1.0,0.0,0.0,0.0,1.0,train


## 3. Export observed data and isolated simulation truth

In [3]:
df.to_csv(DATA/'appointments.csv',index=False)
patient_cols=['patient_id','age','age_band','distance_km','area_deprivation_proxy','vehicle_available','employment','sms_opt_in','preferred_channel']
df[patient_cols].drop_duplicates('patient_id').sort_values('patient_id').to_csv(DATA/'patients.csv',index=False)
df.loc[df.survey_responded.eq(1),['appointment_id','survey_at','attendance_confidence','support_consent',*BARRIERS]].to_csv(DATA/'patient_surveys.csv',index=False)
df.loc[df.active_at_index.eq(1),['appointment_id','index_at','treatment','no_show']].to_csv(DATA/'interventions.csv',index=False)
oracle.to_csv(DATA/'causal_oracle.csv',index=False)
print('Wrote observed data and separately labelled oracle.')

Wrote observed data and separately labelled oracle.


## 4. Exploratory analysis
These statistics describe the synthetic generator, not real NHS prevalence.

In [4]:
active=df.loc[df.active_at_index.eq(1)]
summary=df.outcome.value_counts().rename_axis('outcome').reset_index(name='count')
summary['share']=summary['count']/len(df)
display(summary)
print(f'Binary cohort no-show rate: {active.no_show.mean():.2%}; survey response: {df.survey_responded.mean():.2%}')
barriers=df.loc[df.survey_responded.eq(1),list(BARRIERS)].mean().sort_values(ascending=False)
missing=df.isna().mean().sort_values(ascending=False).rename('missing_share')
monthly=active.assign(month_period=active.scheduled_at.str[:7]).groupby('month_period').no_show.agg(['mean','size'])
summary.to_csv(REPORTS/'outcomes.csv',index=False); barriers.to_csv(REPORTS/'barrier_prevalence.csv'); missing.to_csv(REPORTS/'missingness.csv'); monthly.to_csv(REPORTS/'monthly_attendance.csv')
fig,ax=plt.subplots(figsize=(8,3.8)); ax.bar(summary.outcome,summary['count']); ax.set(title='Synthetic appointment dispositions',ylabel='Appointments'); fig.tight_layout(); fig.savefig(REPORTS/'outcomes.png',dpi=150); plt.show(); plt.close(fig)
fig,ax=plt.subplots(figsize=(8,3.8)); (barriers*100).sort_values().plot.barh(ax=ax); ax.set(title='Reported attendance barriers among survey responders',xlabel='Percentage of responders'); fig.tight_layout(); fig.savefig(REPORTS/'barriers.png',dpi=150); plt.show(); plt.close(fig)
fig,ax=plt.subplots(figsize=(9,3.8)); ax.plot(monthly.index,monthly['mean']*100); ax.set(title='Monthly synthetic no-show rate (active cohort)',ylabel='No-show %'); ax.set_xticks(range(0,len(monthly),3)); ax.tick_params(axis='x',rotation=45); fig.tight_layout(); fig.savefig(REPORTS/'monthly_rate.png',dpi=150); plt.show(); plt.close(fig)

,outcome,count,share
0,attended,37286,0.74572
1,no_show,8390,0.16780
2,cancelled,2790,0.05580
3,rescheduled,1534,0.03068


Binary cohort no-show rate: 18.37%; survey response: 80.10%


/tmp/ipykernel_1944/1095658304.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig,ax=plt.subplots(figsize=(8,3.8)); ax.bar(summary.outcome,summary['count']); ax.set(title='Synthetic appointment dispositions',ylabel='Appointments'); fig.tight_layout(); fig.savefig(REPORTS/'outcomes.png',dpi=150); plt.show(); plt.close(fig)
/tmp/ipykernel_1944/1095658304.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig,ax=plt.subplots(figsize=(8,3.8)); (barriers*100).sort_values().plot.barh(ax=ax); ax.set(title='Reported attendance barriers among survey responders',xlabel='Percentage of responders'); fig.tight_layout(); fig.savefig(REPORTS/'barriers.png',dpi=150); plt.show(); plt.close(fig)
/tmp/ipykernel_1944/1095658304.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig,ax=plt.subplots(figsize=(9,3.8)); ax.plot(monthly.index,monthly['mean']*100); ax.set(title='Monthly synthetic no-show rate (act

## 5. Chronological splits and leakage-safe feature engineering
Train and validation labels must be available before the next period begins. The test period is held out. Patient/appointment IDs remain in raw exports for traceability but never enter the feature pipeline. Use the same splits for both models. Repeated patients across periods are intentional; report cold-start performance in Phase 3. The outcome is observed under the historical simulated offer policy, not untreated risk.

In [5]:
parts=splits(df)
rows=[]
for name,part in parts.items():
    part.to_csv(DATA/'splits'/f'{name}.csv',index=False)
    rows.append({'split':name,'rows':len(part),'patients':part.patient_id.nunique(),'start':part.index_at.min(),'end':part.index_at.max(),'no_show_rate':float(part.no_show.mean())})
split_summary=pd.DataFrame(rows); display(split_summary); split_summary.to_csv(REPORTS/'split_summary.csv',index=False)
overlap={f'{a}_{b}':len(set(parts[a].patient_id)&set(parts[b].patient_id)) for a,b in [('train','validation'),('train','test'),('validation','test')]}
print('Patient overlap:',overlap)
feature_shapes={}
for variant,features in [('baseline',BASE),('survey',FEATURES)]:
    pipeline=preprocessor(variant=='survey')
    matrices={'train':pipeline.fit_transform(parts['train'][features])}
    for split in ('validation','test'): matrices[split]=pipeline.transform(parts[split][features])
    assert all(np.isfinite(x).all() for x in matrices.values())
    assert len({x.shape[1] for x in matrices.values()})==1
    feature_shapes[variant]={k:list(x.shape) for k,x in matrices.items()}
    joblib.dump(pipeline,MODELS/f'phase2_{variant}_preprocessor.joblib',compress=3)
    pd.Series(pipeline.get_feature_names_out(),name='feature').to_csv(MODELS/f'phase2_{variant}_feature_names.csv',index=False)
    np.savez_compressed(MODELS/f'phase2_{variant}_matrices.npz',**{f'X_{k}':x.astype('float32') for k,x in matrices.items()},**{f'y_{k}':parts[k].no_show.to_numpy(dtype='int8') for k in matrices})
print(json.dumps(feature_shapes,indent=2))

,split,rows,patients,start,end,no_show_rate
0,train,30853,11937,2022-12-31T08:00:00Z,2025-01-01T16:30:00Z,0.179302
1,validation,7501,5822,2025-01-04T08:00:00Z,2025-07-02T16:30:00Z,0.192908
2,test,7322,5739,2025-07-05T08:00:00Z,2025-12-24T16:30:00Z,0.192707


Patient overlap: {'train_validation': 5424, 'train_test': 5351, 'validation_test': 2499}


{
  "baseline": {
    "train": [
      30853,
      33
    ],
    "validation": [
      7501,
      33
    ],
    "test": [
      7322,
      33
    ]
  },
  "survey": {
    "train": [
      30853,
      50
    ],
    "validation": [
      7501,
      50
    ],
    "test": [
      7322,
      50
    ]
  }
}


## 6. Simulated intervention ground truth - evaluation only
Observed arm rates are confounded by simulated assignment. Known mean risk reduction versus usual care is calculated from potential risk probabilities on the active cohort; not an estimated real-world treatment effect. The oracle is never used for preprocessing or prediction.

In [6]:
oa=oracle.loc[oracle.active_at_index.eq(1)]
effects=pd.DataFrame([{'arm':a,'assigned_count':int((active.treatment==a).sum()),'observed_no_show_rate':float(active.loc[active.treatment.eq(a),'no_show'].mean()),'oracle_mean_risk':float(oa['p_'+a].mean()),'oracle_risk_reduction_vs_usual_care':float((oa.p_usual_care-oa['p_'+a]).mean()),'minimum_assignment_probability':float(oa['propensity_'+a].min())} for a in ARMS])
display(effects); effects.to_csv(REPORTS/'simulation_effects.csv',index=False)
fig,ax=plt.subplots(figsize=(8,3.8)); ax.bar(effects.arm,effects.oracle_risk_reduction_vs_usual_care*100); ax.set(title='Invented mean risk reduction vs usual care',ylabel='Percentage points'); ax.tick_params(axis='x',rotation=15); fig.tight_layout(); fig.savefig(REPORTS/'simulation_effects.png',dpi=150); plt.show(); plt.close(fig)

,arm,assigned_count,observed_no_show_rate,oracle_mean_risk,oracle_risk_reduction_vs_usual_care,minimum_assignment_probability
0,usual_care,12803,0.212372,0.230700,0.000000,0.138730
1,reminder_offer,13767,0.190310,0.196180,0.034520,0.128516
2,transport_offer,9855,0.155556,0.153552,0.077148,0.104424
3,scheduling_offer,9251,0.164090,0.179273,0.051427,0.104628


/tmp/ipykernel_1944/2183223128.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig,ax=plt.subplots(figsize=(8,3.8)); ax.bar(effects.arm,effects.oracle_risk_reduction_vs_usual_care*100); ax.set(title='Invented mean risk reduction vs usual care',ylabel='Percentage points'); ax.tick_params(axis='x',rotation=15); fig.tight_layout(); fig.savefig(REPORTS/'simulation_effects.png',dpi=150); plt.show(); plt.close(fig)


## 7. Integrate and validate SQLite database
The Phase 1 snapshot remains intact. The integrated database uses the same Phase 1 schema plus three research tables; historical hypothetical offers never create operational interventions or SMS records.

In [7]:
created=integrate(DB,df,oracle,SEED)
db_checks=verify_database(DB,df,oracle)
assert all(db_checks.values()),db_checks
assert not integrate(DB,df,oracle,SEED), 'Repeat should be a no-op'
with connect(DB) as c:
    db_info=integrity(c)
    table_counts={t:c.execute(f'SELECT count(*) FROM "{t}"').fetchone()[0] for t in ('patients','appointments','surveys','barriers','research_runs','research_appointments','research_oracle','interventions','communications')}
print('Research import:', 'created' if created else 'already present and verified')
display(pd.DataFrame({'check':list(db_checks),'passed':list(db_checks.values())})); print(db_info); print(table_counts)

Research import: already present and verified


,check,passed
0,database_integrity,True
1,foreign_keys_clean,True
2,schema_extension_tables,True
3,research_appointment_count,True
4,research_patient_count,True
5,survey_count_matches,True
6,research_data_roundtrip,True
7,oracle_roundtrip,True
8,no_live_research_actions,True
9,manifest_hashes_match,True


{'integrity_check': 'ok', 'foreign_key_violations': [], 'schema_version': 1, 'table_count': 17}
{'patients': 12503, 'appointments': 50003, 'surveys': 40055, 'barriers': 41044, 'research_runs': 1, 'research_appointments': 50000, 'research_oracle': 50000, 'interventions': 2, 'communications': 2}


## 8. Patient survey - interactive notebook interface
Use only fictional demo appointment IDs. The editable cell below defaults to a dry run; set `SAVE_DEMO_SURVEY=True` to persist. `None` means unanswered, 0 no and 1 yes. Support consent is separate from SMS consent. Use the fixed fictional clock to make the demo reproducible. Optional widgets require ipywidgets; all backend behaviour is tested regardless of widget availability.

In [8]:
SAVE_DEMO_SURVEY=False
DEMO_APPOINTMENT='A-003'
CONFIDENCE=4
ANSWERS={'transport_difficulty':1,'financial_difficulty':0,'accessibility_need':None,'timing_conflict':1,'language_support':0,'caring_responsibility':None}
SUPPORT_CONSENT=True
DEMO_TIME='2026-11-25T12:00:00Z'
if SAVE_DEMO_SURVEY:
    with connect(DB) as c: sid=submit_survey(c,DEMO_APPOINTMENT,CONFIDENCE,ANSWERS,SUPPORT_CONSENT,now=DEMO_TIME)
    print('Saved fictional survey:',sid)
else:
    with tempfile.TemporaryDirectory(prefix='CarePath survey demo ') as temp:
        demo=initialize(Path(temp)/'demo.db')
        with connect(demo) as c:
            seed_demo(c); sid=submit_survey(c,DEMO_APPOINTMENT,CONFIDENCE,ANSWERS,SUPPORT_CONSENT,now=DEMO_TIME)
            record=dict(c.execute('SELECT * FROM surveys WHERE survey_id=?',(sid,)).fetchone())
        display(pd.DataFrame([record])); print('Dry-run: submitted and verified in temporary database; integrated DB unchanged.')

,survey_id,appointment_id,recorded_at,attendance_confidence,transport_difficulty,financial_difficulty,accessibility_need,timing_conflict,language_support,caring_responsibility,support_consent
0,FORM-7d2522a10e414b03a4c8f7ff6ff9b433,A-003,2026-11-25T12:00:00Z,4,1,0,None,1,0,None,1


Dry-run: submitted and verified in temporary database; integrated DB unchanged.


In [9]:
try:
    import ipywidgets
except ImportError:
    print('Optional ipywidgets unavailable. Use editable survey cell above; install dependencies to enable form.')
else:
    display(survey_widget(DB))

Optional ipywidgets unavailable. Use editable survey cell above; install dependencies to enable form.


## 9. Execute Phase 1 and Phase 2 automated tests

In [10]:
proc=subprocess.run([sys.executable,'-m','unittest','discover','-s','tests','-v'],cwd=ROOT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,encoding='utf-8',errors='replace',env={**os.environ,'PYTHONIOENCODING':'utf-8'})
(REPORTS/'test_results.txt').write_text(proc.stdout,encoding='utf-8')
print(proc.stdout)
if proc.returncode: raise AssertionError('Test failures; see full output above and reports/phase2/test_results.txt')
match=re.search(r'Ran (\d+) tests?',proc.stdout)
assert match, 'Test count not found'
test_count=int(match.group(1)); assert test_count==47, test_count

test_analyst_write_denied (test_foundation.FoundationTests.test_analyst_write_denied) ... Database transaction rolled back: PermissionError
ok
test_audit_immutable (test_foundation.FoundationTests.test_audit_immutable) ... Database transaction rolled back: IntegrityError
Database transaction rolled back: IntegrityError
ok
test_cancelled_appointment_sms_blocked (test_foundation.FoundationTests.test_cancelled_appointment_sms_blocked) ... Database transaction rolled back: IntegrityError
ok
test_config_offline (test_foundation.FoundationTests.test_config_offline) ... ok
test_confirmation_requires_approval (test_foundation.FoundationTests.test_confirmation_requires_approval) ... Database transaction rolled back: ValueError
ok
test_coordinator_cannot_approve (test_foundation.FoundationTests.test_coordinator_cannot_approve) ... Database transaction rolled back: PermissionError
Database transaction rolled back: IntegrityError
ok
test_cross_appointment_message (test_foundation.FoundationTests.t

## 10. Results and validation summary
The report consolidates synthetic dataset quality, chronological partitioning, preprocessing, database integrity and regression tests. Results describe the simulation only; they do not establish real-world clinical effectiveness or intervention benefits.

In [11]:
from datetime import datetime,timezone
report={'release':'0.2.0','status':'PASS','verified_at_utc':datetime.now(timezone.utc).isoformat(),'seed':SEED,'synthetic_only':True,'appointments':len(df),'patients':int(df.patient_id.nunique()),'active_cohort':len(active),'no_show_rate_active':float(active.no_show.mean()),'survey_response_count':int(df.survey_responded.sum()),'survey_response_rate':float(df.survey_responded.mean()),'outcome_counts':{k:int(v) for k,v in df.outcome.value_counts().items()},'splits':rows,'patient_overlap':overlap,'data_checks':checks,'database_checks':db_checks,'database_table_counts':table_counts,'dataset_sha256':dataset_hash(df),'oracle_sha256':dataset_hash(oracle),'preprocessing_shapes':feature_shapes,'tests_passed':test_count,'tests_failed':0,'environment':{'python':platform.python_version(),'platform':platform.platform(),'sqlite':sqlite3.sqlite_version,'numpy':np.__version__,'pandas':pd.__version__,'scikit_learn':sklearn.__version__,'matplotlib':matplotlib.__version__,'joblib':joblib.__version__},'limitations':['Synthetic data and invented intervention effects only','No trained predictive model','No real NHS integration, authentication or clinical validation','Widget UI not verified in build environment; editable form and backend verified','Windows not directly tested']}
(REPORTS/'validation.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
print(f'Data engineering validation completed successfully - {test_count} tests, {len(checks)} dataset checks, {len(db_checks)} database checks.')

Data engineering validation completed successfully - 47 tests, 26 dataset checks, 10 database checks.
